In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Load the dataset named "trump_tweet_sentiment_analysis.csv" using pandas. Ensure the dataset contains at least two columns: "text" and "label".

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/trum_tweet_sentiment_analysis.csv')

# Display the first few rows and columns to verify
print(df.head())
print(df.columns)

                                                text  Sentiment
0  RT @JohnLeguizamo: #trump not draining swamp b...          0
1  ICYMI: Hackers Rig FM Radio Stations To Play A...          0
2  Trump protests: LGBTQ rally in New York https:...          1
3  "Hi I'm Piers Morgan. David Beckham is awful b...          0
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...          0
Index(['text', 'Sentiment'], dtype='object')


# Build a Text Cleaning Pipeline

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Initialize stopwords, lemmatizer, and stemmer once
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)


In [ ]:
def text_cleaning_pipeline(dataset, rule = "lemmatize"):
  """
  This function performs text cleaning steps on a given dataset, including
  lowercasing, removing URLs, emojis, and other special characters,
  removing stopwords, and optionally lemmatizing or stemming the tokens.

  Args:
    dataset (str): The input text to be cleaned.
    rule (str, optional): The rule for token processing. Can be "lemmatize" or "stem".
                          Defaults to "lemmatize".

  Returns:
    str: The cleaned and processed text.
  """
  # Convert the input to small/lower order.
  data = dataset.lower()

  # Remove URLs (http/https and www variations)
  data = re.sub(r'http\S+|www\S+|\w+\.(?:com|org|net|gov|edu|io|co|in)\S*', '', data)
  # Remove mentions (@username) - often found in social media data
  data = re.sub(r'@\w+', '', data)
  # Remove emojis (a basic regex, might not catch all unicode emojis)
  # This regex tries to capture a range of common emoji unicode blocks.
  data = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+', '', data)
  # Remove punctuation and special characters (keep only letters and spaces)
  data = re.sub(r'[^a-z\s]', '', data)
  # Remove multiple spaces
  data = re.sub(r'\s+', ' ', data).strip()

  # Create tokens.
  tokens = data.split()

  # Remove stopwords:
  tokens = [word for word in tokens if word not in stop_words]

  if rule == "lemmatize":
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
  elif rule == "stem":
    tokens = [stemmer.stem(word) for word in tokens]
  else:
    print("Pick between lemmatize or stem")
    return ""

  return " ".join(tokens)

In [ ]:
# Apply the text cleaning pipeline to the 'text' column
df['cleaned_text'] = df['text'].apply(text_cleaning_pipeline)

# Display the first few rows with the new cleaned_text column
print(df[['text', 'cleaned_text', 'Sentiment']].head())

                                                text  \
0  RT @JohnLeguizamo: #trump not draining swamp b...   
1  ICYMI: Hackers Rig FM Radio Stations To Play A...   
2  Trump protests: LGBTQ rally in New York https:...   
3  "Hi I'm Piers Morgan. David Beckham is awful b...   
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...   

                                        cleaned_text  Sentiment  
0  rt trump draining swamp taxpayer dollar trip a...          0  
1  icymi hacker rig fm radio station play antitru...          0  
2    trump protest lgbtq rally new york bbcworld via          1  
3  hi im pier morgan david beckham awful donald t...          0  
4  rt tech firm suing buzzfeed publishing unverif...          0  


# 3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

In [ ]:
from sklearn.model_selection import train_test_split

X = df['cleaned_text']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (1480098,)
Testing data shape: (370025,)


# 4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting to 5000 features as an example

# Fit and transform the training data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform the test data
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")

Shape of X_train_tfidf: (1480098, 5000)
Shape of X_test_tfidf: (370025, 5000)


# 5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Initialize and train the Logistic Regression model
logistic_model = LogisticRegression(max_iter=1000) # Increased max_iter for convergence
logistic_model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred = logistic_model.predict(X_test_tfidf)

# Print the classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.96      0.95    248563
           1       0.91      0.87      0.89    121462

    accuracy                           0.93    370025
   macro avg       0.92      0.91      0.92    370025
weighted avg       0.93      0.93      0.93    370025

